# LeetCode 973 — K Closest Points to Origin bằng Python

Link đề: https://leetcode.com/problems/k-closest-points-to-origin/ · Độ khó: **Medium** · Chủ đề: Array, Math, Divide and Conquer, Geometry, Sorting, Heap (Priority Queue), Quickselect

Skeleton LeetCode:

```python
class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        ...
```

Ghi chú về đáp án: đề cho phép trả về theo **thứ tự bất kỳ**; và khi nhiều điểm **cùng khoảng cách** ở biên (hòa), có thể có **nhiều tập k điểm đúng** (đề nói đáp án duy nhất nhưng ta vẫn kiểm tra tổng quát). Vì vậy notebook **không** so sánh trực tiếp với một đáp án; thay vào đó dùng **hàm kiểm tra tính hợp lệ** `is_valid_k_closest(points, k, res)`: `res` có đúng `k` điểm, là tập con (đa tập) của `points`, và khoảng cách lớn nhất trong `res` **không vượt quá** khoảng cách nhỏ nhất trong phần còn lại. Hàm này được kiểm chứng bằng oracle định nghĩa (thử mọi tập `k` điểm).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn `k` điểm gần gốc `(0, 0)` nhất theo khoảng cách Euclid.
2. Nắm công cụ Python: `sorted(key=...)`, `heapq` (max-heap bằng cách đổi dấu), phân hoạch ba nhánh (quickselect), tìm kiếm nhị phân trên **giá trị khoảng cách**, `Counter` để kiểm tra đa tập.
3. Đi từ **thử mọi tập `k` điểm** → **sắp xếp** → **max-heap kích thước `k`** → **quickselect** → **tìm nhị phân theo khoảng cách bình phương**.
4. Hiểu mấu chốt: so sánh **khoảng cách bình phương** `x² + y²` (số nguyên, không cần `sqrt`, không sai số); chọn `k` nhỏ nhất không cần sắp xếp toàn bộ.
5. Tránh lỗi kinh điển: sắp theo `x` hoặc theo `|x| + |y|`, trả `k` điểm **xa** nhất (heap không đổi dấu), lệch `k`, trả khoảng cách thay vì điểm, xử lý hòa sai trong quickselect (vòng lặp vô hạn), so sánh `sqrt` bằng số thực.

## 2. Đề bài & ví dụ

Cho mảng `points`, mỗi phần tử `[x, y]` là một điểm trên mặt phẳng, và số nguyên `k`. Trả về `k` điểm gần gốc `(0, 0)` nhất. Khoảng cách là khoảng cách Euclid. Trả về theo thứ tự bất kỳ.

**Ví dụ 1:** `points = [[1,3],[-2,2]]`, `k = 1` → `[[-2,2]]` (`√8 < √10`).

**Ví dụ 2:** `points = [[3,3],[5,-1],[-2,4]]`, `k = 2` → `[[3,3],[-2,4]]` (`[[-2,4],[3,3]]` cũng được chấp nhận).

**Ràng buộc:** `1 <= k <= points.length <= 10^4`; `-10^4 <= x, y <= 10^4`.

In [1]:
import bisect
import heapq
import itertools
import math
import random
import time
from collections import Counter
from typing import List

examples = [(([[1, 3], [-2, 2]], 1), [[-2, 2]]), (([[3, 3], [5, -1], [-2, 4]], 2), [[3, 3], [-2, 4]])]
for (pts, k), want in examples:
    print(f"points = {pts}, k = {k} -> {want}")

points = [[1, 3], [-2, 2]], k = 1 -> [[-2, 2]]
points = [[3, 3], [5, -1], [-2, 4]], k = 2 -> [[3, 3], [-2, 4]]


## 3. Kiến thức nền

### 3.1 Khoảng cách bình phương

`√(x² + y²)` tăng theo `x² + y²`, nên **thứ tự** các điểm theo khoảng cách bằng thứ tự theo `d² = x² + y²`. Với `|x|, |y| ≤ 10^4`, `d² ≤ 2·10^8` — số nguyên, tính chính xác, không cần `sqrt`.

In [2]:
def d2(p):
    return p[0] * p[0] + p[1] * p[1]


print([d2(p) for p in [[1, 3], [-2, 2], [3, 3], [5, -1], [-2, 4]]], "| lớn nhất có thể:", d2([10 ** 4, 10 ** 4]))

[10, 8, 18, 26, 20] | lớn nhất có thể: 200000000


### 3.2 Hòa (tie) và nhiều đáp án đúng

Nếu nhiều điểm có cùng khoảng cách ở **biên** thì có thể chọn bất kỳ nhóm con nào của chúng. Ví dụ `points = [[1,0],[0,1],[-1,0]]`, `k = 2`: ba điểm cùng `d² = 1`, nên **ba** tập `k = 2` đều đúng.

In [3]:
pts = [[1, 0], [0, 1], [-1, 0]]
print("các tập k=2 điểm:", [c for c in itertools.combinations(pts, 2)], "| mọi d² =", [d2(p) for p in pts])

các tập k=2 điểm: [([1, 0], [0, 1]), ([1, 0], [-1, 0]), ([0, 1], [-1, 0])] | mọi d² = [1, 1, 1]


### 3.3 Hàm kiểm tra tính hợp lệ

`res` hợp lệ ⇔ (1) `len(res) == k`; (2) mọi điểm của `res` thuộc `points` (tính cả số lần xuất hiện: `Counter`); (3) `max d²(res) ≤ min d²(phần còn lại)` (nếu phần còn lại không rỗng). Thứ tự trong `res` không quan trọng.

In [4]:
def is_valid_k_closest(points, k, res):
    if not isinstance(res, list) or len(res) != k:
        return False
    try:
        c_res = Counter(tuple(p) for p in res)
    except TypeError:
        return False
    if any(len(p) != 2 for p in c_res):
        return False
    c_pts = Counter(tuple(p) for p in points)
    if any(c_res[p] > c_pts[p] for p in c_res):                 # không được lấy điểm không có / lấy quá số lần
        return False
    rest = c_pts - c_res                                        # phần còn lại (đa tập)
    max_in = max(p[0] * p[0] + p[1] * p[1] for p in c_res)
    if not rest:
        return True
    min_out = min(p[0] * p[0] + p[1] * p[1] for p in rest)
    return max_in <= min_out


print(is_valid_k_closest([[1, 0], [0, 1], [-1, 0]], 2, [[1, 0], [0, 1]]), is_valid_k_closest([[1, 0], [0, 1], [5, 5]], 2, [[1, 0], [5, 5]]),
      is_valid_k_closest([[1, 0], [0, 1]], 2, [[1, 0], [1, 0]]))

True False False


## 4. Ý tưởng nền tảng / trực giác

`points = [[3,3],[5,-1],[-2,4]]`, `k = 2`. Tính `d²`: `18`, `26`, `20`. Hai nhỏ nhất là `18` và `20` ⇒ `[3,3]` và `[-2,4]`. Chỉ cần **biết `k` điểm nhỏ nhất theo `d²`**; không cần thứ tự giữa chúng, cũng không cần sắp xếp phần còn lại.

Ba hướng: (a) sắp xếp hết (`O(n log n)`); (b) giữ `k` điểm tốt nhất đang biết bằng **max-heap** (`O(n log k)`); (c) **quickselect** để tách `k` nhỏ nhất (`O(n)` trung bình).

In [5]:
pts = [[3, 3], [5, -1], [-2, 4]]
print([(p, d2(p)) for p in pts], "-> hai nhỏ nhất:", sorted(pts, key=d2)[:2])

[([3, 3], 18), ([5, -1], 26), ([-2, 4], 20)] -> hai nhỏ nhất: [[3, 3], [-2, 4]]


## 5. Brute force — thử mọi tập `k` điểm

**Ý tưởng:** theo đúng định nghĩa, thử mọi tập `k` điểm (theo chỉ số) và nhận tập đầu tiên mà `max d²(trong) ≤ min d²(ngoài)`. `C(n, k)` tập; dùng cho `n ≲ 12` và làm **oracle độc lập**.

In [6]:
def k_closest_brute(points, k):
    n = len(points)
    for idx in itertools.combinations(range(n), k):
        inside = set(idx)
        max_in = max(d2(points[i]) for i in idx)
        outs = [d2(points[i]) for i in range(n) if i not in inside]
        if not outs or max_in <= min(outs):
            return [points[i][:] for i in idx]
    raise AssertionError("không có tập nào hợp lệ?!")


for (pts, k), want in examples:
    got = k_closest_brute(pts, k)
    assert sorted(got) == sorted(want), got
print([k_closest_brute(p, k) for (p, k), _ in examples])

[[[-2, 2]], [[3, 3], [-2, 4]]]


In [7]:
# Code nộp LeetCode cho Brute force (chỉ minh hoạ; C(n, k) tập)
import itertools


class SolutionBrute:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        n = len(points)
        dist = [x * x + y * y for x, y in points]
        for idx in itertools.combinations(range(n), k):
            chosen = set(idx)
            worst = max(dist[i] for i in idx)
            if all(dist[i] >= worst for i in range(n) if i not in chosen):
                return [points[i] for i in idx]
        return []

**Độ phức tạp:** `O(C(n,k) · n)` thời gian — mũ. Với `n = 10^4` không dùng được.

**Nút thắt:** ta thử nhiều tập trong khi chỉ cần `k` giá trị nhỏ nhất theo `d²` — một bài toán "chọn" (selection) chứ không phải "tổ hợp".

## 6. Bước đột phá

Với khoá `d²` cho sẵn, `k` điểm gần nhất chính là `k` phần tử nhỏ nhất theo khoá. Có thể: sắp xếp (`O(n log n)`), heap kích thước `k` (`O(n log k)`), quickselect (`O(n)` trung bình), hoặc tìm nhị phân trên giá trị khoá (`O(n log D)`).

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** sắp xếp (đáp án chính), **Cách 3** max-heap kích thước `k`, **Cách 4** quickselect ba nhánh, **Cách 5** tìm nhị phân theo khoảng cách bình phương.

### Cách 2 — Sắp xếp theo khoảng cách bình phương

**Giải thích:** `sorted(points, key=d2)` rồi lấy `k` phần tử đầu. Ngắn nhất, đủ nhanh cho `n ≤ 10^4`.

In [8]:
def k_closest_sort(points, k):
    return sorted(points, key=d2)[:k]


for (pts, k), want in examples:
    assert is_valid_k_closest(pts, k, k_closest_sort(pts, k)) and sorted(k_closest_sort(pts, k)) == sorted(want)
print([k_closest_sort(p, k) for (p, k), _ in examples])

[[[-2, 2]], [[3, 3], [-2, 4]]]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionSort:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        return sorted(points, key=lambda p: p[0] * p[0] + p[1] * p[1])[:k]

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ (bản sao sắp xếp).

### Cách 3 — Max-heap kích thước `k`

**Ý tưởng:** `heapq` là **min-heap**; để giữ `k` điểm *nhỏ nhất*, ta giữ **max-heap** bằng cách đẩy `-d²`. Đỉnh heap là điểm **xa nhất** trong `k` điểm đang giữ. Với mỗi điểm mới: nếu heap chưa đủ `k` thì đẩy; ngược lại nếu điểm mới **gần hơn** đỉnh thì thay đỉnh (`heapreplace`).

In [10]:
def k_closest_heap(points, k):
    heap = []                                       # phần tử: (-d2, chỉ số) => đỉnh là điểm xa nhất trong k điểm đang giữ
    for i, p in enumerate(points):
        item = (-d2(p), i)
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:                         # -d2 lớn hơn đỉnh <=> d2 nhỏ hơn đỉnh (gần hơn)
            heapq.heapreplace(heap, item)
    return [points[i][:] for _, i in heap]


for (pts, k), want in examples:
    got = k_closest_heap(pts, k)
    assert is_valid_k_closest(pts, k, got) and sorted(got) == sorted(want)
print([k_closest_heap(p, k) for (p, k), _ in examples])

[[[-2, 2]], [[-2, 4], [3, 3]]]


In [11]:
# Code nộp LeetCode cho Cách 3
import heapq


class SolutionHeap:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        heap = []
        for i, (x, y) in enumerate(points):
            item = (-(x * x + y * y), i)
            if len(heap) < k:
                heapq.heappush(heap, item)
            elif item > heap[0]:
                heapq.heapreplace(heap, item)
        return [points[i] for _, i in heap]

**Độ phức tạp:** `O(n log k)` thời gian, `O(k)` bộ nhớ.

### Cách 4 — Quickselect (phân hoạch ba nhánh)

**Ý tưởng:** chọn ngẫu nhiên một pivot theo `d²`, phân hoạch đoạn `[lo, hi]` thành ba vùng: `< pivot`, `== pivot`, `> pivot` (Dutch national flag). Gọi `lt` là chỉ số đầu vùng `==`, `gt` là chỉ số cuối vùng `==`:

- nếu `k < lt`: `k` nhỏ nhất nằm hẳn trong vùng `<` ⇒ `hi = lt - 1`;
- nếu `k > gt + 1`: lấy trọn vùng `<` và `==`, cần thêm từ vùng `>` ⇒ `lo = gt + 1`;
- ngược lại `lt ≤ k ≤ gt + 1`: `k` phần tử đầu đã đúng (vùng `==` toàn khoảng cách bằng nhau) ⇒ dừng.

Phân hoạch ba nhánh **xử lý hòa** gọn: không bao giờ lặp vô hạn khi nhiều khoảng cách bằng nhau.

In [12]:
def k_closest_quickselect(points, k, seed=973):
    pts = [p[:] for p in points]
    rng = random.Random(seed)
    lo, hi = 0, len(pts) - 1
    while lo <= hi:
        pivot = d2(pts[rng.randint(lo, hi)])
        lt, i, gt = lo, lo, hi
        while i <= gt:                                # phân hoạch ba nhánh trên pts[lo..hi]
            d = d2(pts[i])
            if d < pivot:
                pts[lt], pts[i] = pts[i], pts[lt]
                lt += 1
                i += 1
            elif d > pivot:
                pts[i], pts[gt] = pts[gt], pts[i]
                gt -= 1
            else:
                i += 1
        if k < lt:
            hi = lt - 1
        elif k > gt + 1:
            lo = gt + 1
        else:
            break
    return pts[:k]


for (pts, k), want in examples:
    got = k_closest_quickselect(pts, k)
    assert is_valid_k_closest(pts, k, got) and sorted(got) == sorted(want)
print([k_closest_quickselect(p, k) for (p, k), _ in examples])

[[[-2, 2]], [[3, 3], [-2, 4]]]


In [13]:
# Code nộp LeetCode cho Cách 4
import random


class SolutionQuickselect:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        pts = list(points)
        dist = lambda p: p[0] * p[0] + p[1] * p[1]
        lo, hi = 0, len(pts) - 1
        while lo <= hi:
            pivot = dist(pts[random.randint(lo, hi)])
            lt, i, gt = lo, lo, hi
            while i <= gt:
                d = dist(pts[i])
                if d < pivot:
                    pts[lt], pts[i] = pts[i], pts[lt]
                    lt += 1
                    i += 1
                elif d > pivot:
                    pts[i], pts[gt] = pts[gt], pts[i]
                    gt -= 1
                else:
                    i += 1
            if k < lt:
                hi = lt - 1
            elif k > gt + 1:
                lo = gt + 1
            else:
                break
        return pts[:k]

**Độ phức tạp:** `O(n)` trung bình (mỗi vòng kích thước đoạn giảm kỳ vọng một hằng số lần), `O(n²)` xấu nhất cực hiếm (pivot ngẫu nhiên); `O(n)` bộ nhớ (bản sao).

### Cách 5 — Tìm nhị phân theo khoảng cách bình phương

**Ý tưởng:** tìm `t` nhỏ nhất sao cho **số điểm có `d² ≤ t` ≥ `k`**. Khi đó mọi điểm `d² < t` chắc chắn được chọn (đếm là `c_less < k`), và ta lấy thêm `k - c_less` điểm có `d² == t` (xử lý hòa). Mỗi bước đếm `O(n)`; số bước `O(log D)` với `D ≤ 2·10^8` (≈ 28 bước).

In [14]:
def k_closest_bsearch(points, k):
    dist = [d2(p) for p in points]
    lo, hi = 0, max(dist)
    while lo < hi:                                    # t nhỏ nhất với count(d2 <= t) >= k
        mid = (lo + hi) // 2
        if sum(1 for d in dist if d <= mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    t = lo
    res = [points[i][:] for i, d in enumerate(dist) if d < t]
    for i, d in enumerate(dist):
        if len(res) == k:
            break
        if d == t:
            res.append(points[i][:])                  # bổ sung điểm hòa ở biên cho đủ k
    return res


for (pts, k), want in examples:
    got = k_closest_bsearch(pts, k)
    assert is_valid_k_closest(pts, k, got) and sorted(got) == sorted(want)
print([k_closest_bsearch(p, k) for (p, k), _ in examples])

[[[-2, 2]], [[3, 3], [-2, 4]]]


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionBSearch:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        dist = [x * x + y * y for x, y in points]
        lo, hi = 0, max(dist)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum(d <= mid for d in dist) >= k:
                hi = mid
            else:
                lo = mid + 1
        res = [p for p, d in zip(points, dist) if d < lo]
        for p, d in zip(points, dist):
            if len(res) == k:
                break
            if d == lo:
                res.append(p)
        return res

**Độ phức tạp:** `O(n log D)` thời gian, `O(n)` bộ nhớ. Hữu ích khi dữ liệu không nạp hết bộ nhớ (đếm theo từng đợt).

## 8. TRACE từng bước

**Trace Cách 3 (heap)** với `points = [[3,3],[5,-1],[-2,4]]`, `k = 2`:

| điểm | `d²` | heap trước | hành động | heap sau (theo `d²`) |
|---|---|---|---|---|
| `[3,3]` | 18 | `{}` | đẩy | `{18}` |
| `[5,-1]` | 26 | `{18}` | đẩy (chưa đủ `k`) | `{18, 26}` |
| `[-2,4]` | 20 | `{18, 26}` | `20 < 26` ⇒ thay đỉnh | `{18, 20}` |

Kết quả `[[3,3],[-2,4]]`. Hàm trace in từng bước (đỉnh heap = điểm xa nhất):

In [16]:
def trace_heap(points, k):
    heap = []
    print(f"points = {points}, k = {k}")
    for i, p in enumerate(points):
        item = (-d2(p), i)
        before = sorted(-d for d, _ in heap)
        if len(heap) < k:
            heapq.heappush(heap, item)
            act = "đẩy (chưa đủ k)"
        elif item > heap[0]:
            worst = -heap[0][0]
            heapq.heapreplace(heap, item)
            act = f"{d2(p)} < đỉnh {worst}: thay đỉnh"
        else:
            act = f"{d2(p)} >= đỉnh {-heap[0][0]}: bỏ"
        print(f"  {p} d2={d2(p):3d} | heap trước {before} | {act} | heap sau {sorted(-d for d, _ in heap)}")
    res = [points[i] for _, i in heap]
    print("=>", res)
    return res


got = trace_heap([[3, 3], [5, -1], [-2, 4]], 2)
assert sorted(got) == [[-2, 4], [3, 3]]

points = [[3, 3], [5, -1], [-2, 4]], k = 2
  [3, 3] d2= 18 | heap trước [] | đẩy (chưa đủ k) | heap sau [18]
  [5, -1] d2= 26 | heap trước [18] | đẩy (chưa đủ k) | heap sau [18, 26]
  [-2, 4] d2= 20 | heap trước [18, 26] | 20 < đỉnh 26: thay đỉnh | heap sau [18, 20]
=> [[-2, 4], [3, 3]]


**Trace Cách 4 (quickselect)** với các khoảng cách `[5, 1, 5, 3, 5, 2]`, `k = 3`: chọn pivot `5` ⇒ vùng `<` gồm `1, 3, 2` (3 phần tử), vùng `==` gồm ba số `5`. `lt = 3`, `gt = 5`; `k = 3` thoả `lt ≤ k ≤ gt + 1` ⇒ dừng, ba phần tử đầu là `{1, 2, 3}` — đúng. Hàm dưới in từng vòng phân hoạch:

In [17]:
def trace_quickselect(dists, k, seed=1):
    arr = dists[:]
    rng = random.Random(seed)
    lo, hi = 0, len(arr) - 1
    print("khoảng cách:", arr, "k =", k)
    while lo <= hi:
        pivot = arr[rng.randint(lo, hi)]
        lt, i, gt = lo, lo, hi
        while i <= gt:
            if arr[i] < pivot:
                arr[lt], arr[i] = arr[i], arr[lt]
                lt += 1
                i += 1
            elif arr[i] > pivot:
                arr[i], arr[gt] = arr[gt], arr[i]
                gt -= 1
            else:
                i += 1
        print(f"  [lo={lo}, hi={hi}] pivot={pivot}: {arr} | lt={lt}, gt={gt}")
        if k < lt:
            hi = lt - 1
        elif k > gt + 1:
            lo = gt + 1
        else:
            break
    print("=> k nhỏ nhất:", sorted(arr[:k]))
    return sorted(arr[:k])


assert trace_quickselect([5, 1, 5, 3, 5, 2], 3) == [1, 2, 3]
assert trace_quickselect([5, 1, 5, 3, 5, 2], 4) == [1, 2, 3, 5]

khoảng cách: [5, 1, 5, 3, 5, 2] k = 3
  [lo=0, hi=5] pivot=1: [1, 5, 3, 5, 2, 5] | lt=0, gt=0
  [lo=1, hi=5] pivot=5: [1, 3, 2, 5, 5, 5] | lt=3, gt=5
=> k nhỏ nhất: [1, 2, 3]
khoảng cách: [5, 1, 5, 3, 5, 2] k = 4
  [lo=0, hi=5] pivot=1: [1, 5, 3, 5, 2, 5] | lt=0, gt=0
  [lo=1, hi=5] pivot=5: [1, 3, 2, 5, 5, 5] | lt=3, gt=5
=> k nhỏ nhất: [1, 2, 3, 5]


## 9. Code nộp LeetCode

Chọn **Cách 2** (sắp xếp): ngắn, đủ nhanh cho `n ≤ 10^4`, ít lỗi. Giữ nguyên `class Solution`, `kClosest(self, points, k)`:

In [18]:
class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        # khoảng cách bình phương: cùng thứ tự với khoảng cách Euclid, không cần sqrt
        return sorted(points, key=lambda p: p[0] * p[0] + p[1] * p[1])[:k]


print([Solution().kClosest(p, k) for (p, k), _ in examples])
assert all(sorted(Solution().kClosest(p, k)) == sorted(w) for (p, k), w in examples)

[[[-2, 2]], [[3, 3], [-2, 4]]]


## 10. Vì sao đúng

**(a) Khoảng cách bình phương.** Hàm `u ↦ √u` tăng nghiêm ngặt trên `u ≥ 0`, nên `d(p) < d(q) ⇔ d²(p) < d²(q)`: thứ tự theo khoảng cách bằng thứ tự theo `d²`. `d²` là số nguyên nên so sánh chính xác.

**(b) Tập `k` điểm là đáp án ⇔ `max d²(trong) ≤ min d²(ngoài)`.** (⇒) Nếu có điểm ngoài gần hơn điểm trong xa nhất thì thay thế được, tập không phải `k` gần nhất. (⇐) Nếu thoả điều kiện, mọi điểm trong có `d²` ≤ mọi điểm ngoài, nên là một lựa chọn `k` nhỏ nhất (có thể khác nhau ở các điểm hòa, đều đúng).

**(c) Cách 2.** Sau khi sắp xếp không giảm theo `d²`, `k` phần tử đầu thoả (b): mọi phần tử đầu `≤` mọi phần tử sau.

**(d) Cách 3.** Bất biến: sau khi xử lý `i` điểm đầu, heap chứa `min(k, i)` điểm nhỏ nhất (theo `d²`) trong `i` điểm đó. Điểm mới `p`: nếu heap chưa đủ `k` thì thêm vào (vẫn là tập nhỏ nhất); nếu đủ `k` và `d²(p)` nhỏ hơn `d²` của điểm xa nhất đang giữ thì thay điểm xa nhất bằng `p`, giữ bất biến; nếu không thì `p` không thuộc `k` nhỏ nhất (đã có `k` điểm không xa hơn). Đỉnh heap lưu `-d²` nên là điểm xa nhất.

**(e) Cách 4.** Bất biến: mọi phần tử ở chỉ số `< lo` có `d² ≤` mọi phần tử trong `[lo, hi]`, và mọi phần tử ở chỉ số `> hi` có `d² ≥` chúng. Sau phân hoạch ba nhánh: vùng `<` ở đầu, vùng `==`, vùng `>` ở cuối. `k` nhỏ nhất nằm trọn trong `[0, k)`: nếu `k < lt`, nó nằm trong vùng `<` ⇒ thu hẹp `hi = lt - 1`; nếu `k > gt + 1`, vùng `<` và `==` đều được chọn, cần thêm từ vùng `>` ⇒ `lo = gt + 1`; nếu `lt ≤ k ≤ gt + 1` thì `[0, k)` gồm toàn bộ vùng `<` cộng một số phần tử vùng `==` ⇒ đúng (b). Vùng `==` luôn không rỗng (chứa pivot) nên mỗi vòng thu hẹp đoạn hoặc dừng: kết thúc.

**(f) Cách 5.** Hàm đếm `c(t) = #{d² ≤ t}` không giảm theo `t`. `t*` nhỏ nhất với `c(t*) ≥ k`. Khi đó `c(t* - 1) < k` nghĩa là số điểm `d² < t*` nhỏ hơn `k`, nên tất cả chúng đều được chọn; `c(t*) ≥ k` nghĩa là còn đủ điểm `d² == t*` để lấp phần thiếu. Tập kết quả thoả (b): trong ≤ `t*` và ngoài ≥ `t*`.

**(g) Kiểm chứng bằng máy.** Với **mọi** danh sách ≤ 4 điểm trên lưới `{-1,0,1}²` (kể cả điểm trùng và hòa) và **mọi** `k`, mọi cách trả về kết quả hợp lệ theo **oracle định nghĩa** (thử mọi tập `k` điểm); hàm `is_valid_k_closest` cũng được kiểm chứng trùng khớp oracle đó trên mọi tập con (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi tập `k` điểm | `O(C(n,k)·n)` | `O(n)` | oracle `n ≲ 12` |
| 2. Sắp xếp | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. Max-heap kích thước `k` | `O(n log k)` | `O(k)` | `k` nhỏ, dữ liệu dòng |
| 4. Quickselect | `O(n)` trung bình | `O(n)` | nhanh nhất |
| 5. Tìm nhị phân khoảng cách | `O(n log D)` | `O(n)` | đếm từng đợt |

Đo thực tế `n = 10^4`, `k = 100` và `k = 5000`:

In [19]:
def bench(fn, args, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(973)
big = [[rng.randint(-10 ** 4, 10 ** 4), rng.randint(-10 ** 4, 10 ** 4)] for _ in range(10 ** 4)]
for k in (100, 5000):
    print(f"k = {k}")
    for name, fn in [("sort", k_closest_sort), ("heap", k_closest_heap), ("quickselect", k_closest_quickselect), ("bsearch", k_closest_bsearch)]:
        print(f"  {name:12s}: {bench(fn, (big, k)) * 1e3:7.2f} ms")

k = 100
  sort        :    2.43 ms
  heap        :    1.84 ms


  quickselect :    7.20 ms


  bsearch     :    7.33 ms
k = 5000
  sort        :    2.41 ms


  heap        :    4.76 ms


  quickselect :    7.50 ms


  bsearch     :   11.37 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn lưới nhỏ (kể cả hòa và điểm trùng), (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

Vì có thể nhiều đáp án đúng, mọi so sánh dùng `is_valid_k_closest`; ngoài ra, khi **khoảng cách `k` nhỏ nhất đôi một khác nhau** (không hòa ở biên) đáp án là duy nhất nên ta so sánh `sorted` trực tiếp.

### A. Ví dụ của đề

In [20]:
IMPLS = {
    "sort": k_closest_sort,
    "heap": k_closest_heap,
    "quickselect": k_closest_quickselect,
    "bsearch": k_closest_bsearch,
    "brute": k_closest_brute,
    "Solution": lambda p, k: Solution().kClosest([q[:] for q in p], k),
    "SolutionSort": lambda p, k: SolutionSort().kClosest([q[:] for q in p], k),
    "SolutionHeap": lambda p, k: SolutionHeap().kClosest([q[:] for q in p], k),
    "SolutionQuickselect": lambda p, k: SolutionQuickselect().kClosest([q[:] for q in p], k),
    "SolutionBSearch": lambda p, k: SolutionBSearch().kClosest([q[:] for q in p], k),
    "SolutionBrute": lambda p, k: SolutionBrute().kClosest([q[:] for q in p], k),
}
SLOW = {"brute", "SolutionBrute"}


def unique_answer(points, k):
    """Nếu k nhỏ nhất theo d2 là duy nhất (không hòa ở biên) trả về tập đó (đã sort), ngược lại None."""
    ds = sorted(d2(p) for p in points)
    if k < len(ds) and ds[k - 1] == ds[k]:
        return None
    thr = ds[k - 1]
    chosen = sorted(p for p in points if d2(p) <= thr)
    return chosen if len(chosen) == k else None


def check(points, k, names=None):
    for name in (names or IMPLS):
        if name in SLOW and len(points) > 10:
            continue
        pts_in = [p[:] for p in points]
        got = IMPLS[name](pts_in, k)
        assert is_valid_k_closest(points, k, got), (name, points, k, got)
        uniq = unique_answer(points, k)
        if uniq is not None:
            assert sorted(got) == uniq, (name, points, k, got, uniq)


for (pts, k), want in examples:
    check(pts, k)
    assert sorted(Solution().kClosest(pts, k)) == sorted(want)
print("ví dụ đề: brute và", len(IMPLS), "cài đặt cho kết quả hợp lệ; đáp án duy nhất khớp")

ví dụ đề: brute và 11 cài đặt cho kết quả hợp lệ; đáp án duy nhất khớp


### B. Edge case

- `k = n`: trả lại **tất cả** điểm (mọi thứ tự).
- `k = 1`, `n = 1`.
- Điểm trùng nhau (nhiều bản sao cùng toạ độ).
- Toàn bộ điểm cùng khoảng cách (hòa hoàn toàn): `[[1,0],[0,1],[-1,0],[0,-1]]`.
- Gốc `(0, 0)` nằm trong danh sách.
- Toạ độ cực đoan `±10^4`.
- Điểm khác nhau nhưng cùng `d²` ở biên (`[3,4]` và `[5,0]` cùng `d² = 25`).
- `k` nhỏ nhất và lớn nhất.

In [21]:
edge = [
    ([[5, 5]], 1), ([[1, 2], [3, 4], [5, 6]], 3), ([[0, 0]], 1), ([[0, 0], [0, 0], [1, 1]], 2), ([[0, 0], [0, 0], [1, 1]], 1),
    ([[1, 0], [0, 1], [-1, 0], [0, -1]], 1), ([[1, 0], [0, 1], [-1, 0], [0, -1]], 2), ([[1, 0], [0, 1], [-1, 0], [0, -1]], 4),
    ([[3, 4], [5, 0], [0, 5], [4, 3], [1, 1]], 2), ([[3, 4], [5, 0], [0, 5], [4, 3], [1, 1]], 3),
    ([[10 ** 4, 10 ** 4], [-10 ** 4, -10 ** 4], [10 ** 4, -10 ** 4], [0, 10 ** 4]], 2),
    ([[2, 2], [2, 2], [2, 2], [1, 1]], 3), ([[-1, -1], [1, 1], [1, -1], [-1, 1]], 3), ([[7, 0], [0, 7], [7, 0]], 2),
    ([[1, 3], [-2, 2]], 1), ([[3, 3], [5, -1], [-2, 4]], 2),
]
for pts, k in edge:
    check(pts, k)
print(len(edge), "edge case: mọi cài đặt cho kết quả hợp lệ")
print("hòa hoàn toàn k=2 ->", Solution().kClosest([[1, 0], [0, 1], [-1, 0], [0, -1]], 2))

16 edge case: mọi cài đặt cho kết quả hợp lệ
hòa hoàn toàn k=2 -> [[1, 0], [0, 1]]


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [22]:
def bug_sort_by_x(points, k):                         # sắp theo x thay vì khoảng cách
    return sorted(points, key=lambda p: p[0])[:k]


def bug_manhattan(points, k):                         # dùng |x| + |y| (khoảng cách Manhattan)
    return sorted(points, key=lambda p: abs(p[0]) + abs(p[1]))[:k]


def bug_farthest(points, k):                          # trả k điểm XA nhất
    return sorted(points, key=d2, reverse=True)[:k]


def bug_off_by_one(points, k):                        # lấy k - 1 điểm
    return sorted(points, key=d2)[:k - 1]


def bug_heap_min(points, k):                          # heap không đổi dấu: giữ k điểm xa nhất
    heap = []
    for i, p in enumerate(points):
        item = (d2(p), i)
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
    return [points[i] for _, i in heap]


def bug_distances(points, k):                         # trả khoảng cách thay vì điểm
    return sorted(d2(p) for p in points)[:k]


def bug_x_squared_minus_y_squared(points, k):         # x*x - y*y
    return sorted(points, key=lambda p: p[0] * p[0] - p[1] * p[1])[:k]


def bug_distinct_only(points, k):                     # khử trùng điểm trước khi chọn (mất bản sao)
    uniq = [list(t) for t in sorted({tuple(p) for p in points}, key=lambda t: d2(t))]
    return uniq[:k]


BUGS = [bug_sort_by_x, bug_manhattan, bug_farthest, bug_off_by_one, bug_heap_min, bug_distances, bug_x_squared_minus_y_squared, bug_distinct_only]
probe = [([[3, 3], [5, -1], [-2, 4]], 2), ([[1, 3], [-2, 2]], 1), ([[0, 0], [0, 0], [1, 1]], 2), ([[5, 0], [3, 3], [1, 4], [-2, -2], [4, 4]], 3),
         ([[3, 4], [-4, 0], [6, 0], [0, -5]], 2), ([[1, 1], [2, 0], [-1, -1], [0, 3]], 3)]
for bug in BUGS:
    found = None
    for pts, k in probe:
        try:
            got = bug([p[:] for p in pts], k)
            ok = is_valid_k_closest(pts, k, got)
        except Exception as e:
            got, ok = type(e).__name__, False
        if not ok:
            found = (pts, k, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:32s} sai trên points={found[0]}, k={found[1]}: ra {found[2]}")

bug_sort_by_x                    sai trên points=[[1, 1], [2, 0], [-1, -1], [0, 3]], k=3: ra [[-1, -1], [0, 3], [1, 1]]
bug_manhattan                    sai trên points=[[3, 3], [5, -1], [-2, 4]], k=2: ra [[3, 3], [5, -1]]
bug_farthest                     sai trên points=[[3, 3], [5, -1], [-2, 4]], k=2: ra [[5, -1], [-2, 4]]
bug_off_by_one                   sai trên points=[[3, 3], [5, -1], [-2, 4]], k=2: ra [[3, 3]]
bug_heap_min                     sai trên points=[[3, 3], [5, -1], [-2, 4]], k=2: ra [[-2, 4], [5, -1]]
bug_distances                    sai trên points=[[3, 3], [5, -1], [-2, 4]], k=2: ra [18, 20]
bug_x_squared_minus_y_squared    sai trên points=[[1, 3], [-2, 2]], k=1: ra [[1, 3]]
bug_distinct_only                sai trên points=[[0, 0], [0, 0], [1, 1]], k=2: ra [[0, 0], [1, 1]]


### D. Vét cạn lưới nhỏ (kể cả hòa và điểm trùng)

Lưới `G = {-1, 0, 1}²` (9 điểm). Các bước:

1. **Chứng minh hàm kiểm tra đúng**: với mỗi danh sách ≤ 4 điểm và mỗi `k`, và **mỗi tập `k` chỉ số**, `is_valid_k_closest` bằng oracle định nghĩa `max_in ≤ min_out`.
2. **Mọi danh sách** `n ≤ 4` (có lặp) × mọi `k ∈ [1, n]`: mọi cài đặt cho kết quả hợp lệ và (nếu không hòa ở biên) khớp đáp án duy nhất.
3. **Mẫu `n = 5`**.

In [23]:
G = [[x, y] for x in (-1, 0, 1) for y in (-1, 0, 1)]
t0 = time.perf_counter()
cases = 0
checked_valid_fn = 0
for n in range(1, 5):
    for pts in itertools.product(G, repeat=n):
        pts = [list(p) for p in pts]
        for k in range(1, n + 1):
            # (1) hàm kiểm tra khớp oracle định nghĩa trên MỌI tập k chỉ số
            if n <= 3:
                for idx in itertools.combinations(range(n), k):
                    inside = set(idx)
                    max_in = max(d2(pts[i]) for i in idx)
                    outs = [d2(pts[i]) for i in range(n) if i not in inside]
                    want = (not outs) or max_in <= min(outs)
                    assert is_valid_k_closest(pts, k, [pts[i][:] for i in idx]) == want, (pts, k, idx)
                    checked_valid_fn += 1
            # (2) mọi cài đặt
            for name in IMPLS:
                if name in SLOW and n > 4:
                    continue
                got = IMPLS[name]([p[:] for p in pts], k)
                assert is_valid_k_closest(pts, k, got), (name, pts, k, got)
                uniq = unique_answer(pts, k)
                if uniq is not None:
                    assert sorted(got) == uniq, (name, pts, k)
            cases += 1
print(cases, "ca (danh sách ≤ 4 điểm trên lưới 3×3, mọi k);", checked_valid_fn, "lần kiểm hàm kiểm tra vs oracle; mọi cài đặt hợp lệ;", round(time.perf_counter() - t0, 1), "s")

28602 ca (danh sách ≤ 4 điểm trên lưới 3×3, mọi k); 5355 lần kiểm hàm kiểm tra vs oracle; mọi cài đặt hợp lệ; 4.2 s


In [24]:
rng = random.Random(97301)
t0 = time.perf_counter()
fast = [n for n in IMPLS if n not in SLOW]
cnt = 0
for _ in range(3000):
    n = 5
    pts = [list(rng.choice(G)) for _ in range(n)]
    for k in range(1, n + 1):
        for name in fast:
            got = IMPLS[name]([p[:] for p in pts], k)
            assert is_valid_k_closest(pts, k, got), (name, pts, k, got)
        if k <= 3:
            assert is_valid_k_closest(pts, k, k_closest_brute(pts, k))
        cnt += 1
print("n = 5:", cnt, "ca mẫu: hợp lệ;", round(time.perf_counter() - t0, 1), "s")

n = 5: 15000 ca mẫu: hợp lệ; 1.8 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Kết quả hợp lệ theo `is_valid_k_closest`; (2) `sorted(d²)` của kết quả **bằng** `k` giá trị nhỏ nhất của `sorted(d²)` của toàn bộ (độc lập với việc chọn điểm hòa nào); (3) trả về bản sao hay tham chiếu không làm hỏng dữ liệu đầu vào (danh sách `points` sau gọi vẫn là hoán vị của bản gốc); (4) hoán vị đầu vào không đổi đa tập khoảng cách của kết quả; (5) `k = n` ⇒ kết quả là hoán vị của đầu vào.

In [25]:
rng = random.Random(20240973)
t0 = time.perf_counter()
for it in range(400):
    n = rng.randint(1, 300)
    span = rng.choice([2, 5, 30, 10 ** 4])                   # span nhỏ => nhiều hòa và điểm trùng
    pts = [[rng.randint(-span, span), rng.randint(-span, span)] for _ in range(n)]
    k = rng.randint(1, n)
    want_d = sorted(d2(p) for p in pts)[:k]
    for name in fast:
        orig = [p[:] for p in pts]
        got = IMPLS[name](orig, k)
        assert is_valid_k_closest(pts, k, got), (name, it)                    # (1)
        assert sorted(d2(p) for p in got) == want_d, (name, it)               # (2)
        assert sorted(map(tuple, orig)) == sorted(map(tuple, pts)), name      # (3)
    sh = pts[:]
    rng.shuffle(sh)
    assert sorted(d2(p) for p in Solution().kClosest(sh, k)) == want_d        # (4)
    assert sorted(map(tuple, Solution().kClosest(pts, n))) == sorted(map(tuple, pts))   # (5)
print("400 ca ngẫu nhiên (n ≤ 300, nhiều hòa): 5 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

400 ca ngẫu nhiên (n ≤ 300, nhiều hòa): 5 tính chất đúng; 0.9 s


In [26]:
# Nhiều hòa ở biên: tập có khoảng cách lặp lại; kiểm tra heap/quickselect/bsearch không treo và hợp lệ
rng = random.Random(97302)
t0 = time.perf_counter()
for _ in range(300):
    base = [[rng.randint(-3, 3), rng.randint(-3, 3)] for _ in range(rng.randint(1, 12))]
    pts = base * rng.randint(1, 5)                              # nhiều bản sao
    rng.shuffle(pts)
    for k in sorted({1, len(pts), max(1, len(pts) // 2), rng.randint(1, len(pts))}):
        for name in fast:
            assert is_valid_k_closest(pts, k, IMPLS[name]([p[:] for p in pts], k)), (name, k)
        if len(pts) <= 10:
            assert is_valid_k_closest(pts, k, k_closest_brute(pts, k))
print("300 danh sách lặp bản sao: hợp lệ;", round(time.perf_counter() - t0, 1), "s")

300 danh sách lặp bản sao: hợp lệ; 0.2 s


### F. Giới hạn

`n = 10^4`, `k` ở các biên (`1`, `n`, `n/2`), toạ độ `±10^4`, và trường hợp **mọi điểm trùng nhau** (hòa tuyệt đối) cho quickselect.

In [27]:
t0 = time.perf_counter()
rng = random.Random(97303)
big = [[rng.randint(-10 ** 4, 10 ** 4), rng.randint(-10 ** 4, 10 ** 4)] for _ in range(10 ** 4)]
for k in (1, 2, 100, 5000, 9999, 10 ** 4):
    want_d = sorted(d2(p) for p in big)[:k]
    for name in fast:
        got = IMPLS[name]([p[:] for p in big], k)
        assert len(got) == k and sorted(d2(p) for p in got) == want_d, (name, k)
        assert is_valid_k_closest(big, k, got), (name, k)
same = [[7, 7]] * (10 ** 4)
for k in (1, 5000, 10 ** 4):
    for name in fast:
        got = IMPLS[name]([p[:] for p in same], k)
        assert got == [[7, 7]] * k, (name, k)
corners = [[10 ** 4, 10 ** 4], [-10 ** 4, 10 ** 4], [10 ** 4, -10 ** 4], [-10 ** 4, -10 ** 4]] * 2500
for name in fast:
    assert is_valid_k_closest(corners, 4000, IMPLS[name]([p[:] for p in corners], 4000)), name
print("n = 10^4 (6 giá trị k, toạ độ ±10^4), mọi điểm trùng, bốn góc lặp lại: mọi cài đặt đúng;", round(time.perf_counter() - t0, 1), "s")

n = 10^4 (6 giá trị k, toạ độ ±10^4), mọi điểm trùng, bốn góc lặp lại: mọi cài đặt đúng; 1.0 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sắp xếp theo `x` hoặc theo `|x| + |y|`

In [28]:
pts_x = [[-9, 0], [1, 1], [2, 2]]
pts_m = [[3, 3], [5, 0], [4, 4]]
print("theo x:", bug_sort_by_x(pts_x, 1), "| đúng:", Solution().kClosest(pts_x, 1))
print("Manhattan:", bug_manhattan(pts_m, 1), "| đúng:", Solution().kClosest(pts_m, 1))
assert not is_valid_k_closest(pts_x, 1, bug_sort_by_x(pts_x, 1))
assert not is_valid_k_closest(pts_m, 1, bug_manhattan(pts_m, 1))

theo x: [[-9, 0]] | đúng: [[1, 1]]
Manhattan: [[5, 0]] | đúng: [[3, 3]]


**Vì sao sai:** khoảng cách Euclid là `√(x² + y²)`; `x` một mình hay `|x| + |y|` có thứ tự khác (`(3,3)` có Manhattan `6` nhưng Euclid `√18 ≈ 4.24 < 5`). **Đúng:** `x*x + y*y`.

### Lỗi 2: trả `k` điểm xa nhất (heap không đổi dấu)

In [29]:
pts = [[3, 3], [5, -1], [-2, 4]]
print("bug:", bug_heap_min(pts, 2), "| farthest:", bug_farthest(pts, 2), "| đúng:", Solution().kClosest(pts, 2))
assert not is_valid_k_closest(pts, 2, bug_heap_min(pts, 2)) and not is_valid_k_closest(pts, 2, bug_farthest(pts, 2))

bug: [[-2, 4], [5, -1]] | farthest: [[5, -1], [-2, 4]] | đúng: [[3, 3], [-2, 4]]


**Vì sao sai:** `heapq` là min-heap. Muốn giữ `k` điểm **nhỏ nhất** phải đẩy `-d²` để đỉnh là điểm **xa nhất** trong số đang giữ (để bị thay thế).

### Lỗi 3: lệch `k` hoặc trả khoảng cách thay vì điểm

In [30]:
pts = [[3, 3], [5, -1], [-2, 4]]
print("k-1:", bug_off_by_one(pts, 2), "| khoảng cách:", bug_distances(pts, 2), "| đúng:", Solution().kClosest(pts, 2))
assert len(bug_off_by_one(pts, 2)) == 1 and not is_valid_k_closest(pts, 2, bug_distances(pts, 2))

k-1: [[3, 3]] | khoảng cách: [18, 20] | đúng: [[3, 3], [-2, 4]]


**Vì sao sai:** đề yêu cầu **đúng `k` điểm** (mỗi điểm là `[x, y]`), không phải số khoảng cách.

### Lỗi 4: khử trùng điểm làm mất bản sao

In [31]:
pts = [[0, 0], [0, 0], [1, 1]]
print("bug:", bug_distinct_only(pts, 2), "| đúng:", Solution().kClosest(pts, 2))
assert not is_valid_k_closest(pts, 2, bug_distinct_only(pts, 2))

bug: [[0, 0], [1, 1]] | đúng: [[0, 0], [0, 0]]


**Vì sao sai:** hai điểm trùng toạ độ vẫn là hai điểm khác nhau trong đầu vào; đề không cho phép bỏ bản sao. Đây cũng là lý do kiểm tra tính hợp lệ dùng `Counter` (đa tập).

### Lỗi 5: sai dấu trong công thức khoảng cách

In [32]:
pts = [[1, 1], [2, 0], [-1, -1], [0, 3]]
print("x²-y²:", bug_x_squared_minus_y_squared(pts, 3), "| đúng:", Solution().kClosest(pts, 3))
assert not is_valid_k_closest(pts, 3, bug_x_squared_minus_y_squared(pts, 3))

x²-y²: [[0, 3], [1, 1], [-1, -1]] | đúng: [[1, 1], [-1, -1], [2, 0]]


**Vì sao sai:** `x² − y²` có thể âm và không phải khoảng cách. **Đúng:** `x² + y²`.

### Lỗi 6 (bonus): quickselect không xử lý hòa

In [33]:
def comparisons_two_way(dists, k):
    """Quickselect hai nhánh kiểu Lomuto (pivot = phần tử cuối): đếm số lần so sánh."""
    arr, lo, hi, comps = dists[:], 0, len(dists) - 1, 0
    while lo < hi:
        pivot, i = arr[hi], lo
        for j in range(lo, hi):
            comps += 1
            if arr[j] < pivot:
                arr[i], arr[j] = arr[j], arr[i]
                i += 1
        arr[i], arr[hi] = arr[hi], arr[i]
        if i == k:
            break
        lo, hi = (i + 1, hi) if i < k else (lo, i - 1)
    return comps


def comparisons_three_way(dists, k):
    arr, lo, hi, comps = dists[:], 0, len(dists) - 1, 0
    rng = random.Random(1)
    while lo <= hi:
        pivot = arr[rng.randint(lo, hi)]
        lt, i, gt = lo, lo, hi
        while i <= gt:
            comps += 1
            if arr[i] < pivot:
                arr[lt], arr[i] = arr[i], arr[lt]
                lt += 1
                i += 1
            elif arr[i] > pivot:
                arr[i], arr[gt] = arr[gt], arr[i]
                gt -= 1
            else:
                i += 1
        if k < lt:
            hi = lt - 1
        elif k > gt + 1:
            lo = gt + 1
        else:
            break
    return comps


same = [25] * 2000
c2, c3 = comparisons_two_way(same, 1000), comparisons_three_way(same, 1000)
print("2000 khoảng cách bằng nhau, k = 1000: hai nhánh", c2, "so sánh | ba nhánh", c3, "so sánh")
assert c2 > 100 * c3
pts = [[1, 0]] * 40
assert is_valid_k_closest(pts, 20, k_closest_quickselect(pts, 20))

2000 khoảng cách bằng nhau, k = 1000: hai nhánh 1500499 so sánh | ba nhánh 2000 so sánh


**Vì sao sai:** khi nhiều khoảng cách bằng pivot, phân hoạch hai nhánh chỉ loại được **một** phần tử mỗi vòng ⇒ `O(n·k)` so sánh thay vì `O(n)` (và các biến thể viết ẩu còn có thể lặp vô hạn). **Đúng:** phân hoạch ba nhánh gom toàn bộ phần tử bằng pivot trong một lần quét.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chọn `k` phần tử nhỏ/lớn nhất theo một khoá" → sắp xếp / heap kích thước `k` / quickselect; khi `k` nhỏ so với `n` dùng heap, khi chỉ cần tập (không cần thứ tự) dùng quickselect.

**Mẹo:**

1. So sánh **khoảng cách bình phương** (số nguyên).
2. Muốn `k` **nhỏ nhất** dùng **max-heap** (`-d²`); muốn `k` **lớn nhất** dùng min-heap.
3. Hòa ở biên ⇒ nhiều đáp án; kiểm tra bằng hàm hợp lệ, không so từng phần tử.
4. Quickselect: phân hoạch **ba nhánh** để chịu được nhiều giá trị bằng nhau.
5. Tìm nhị phân trên **giá trị** khi có thể đếm nhanh "bao nhiêu phần tử ≤ t".
6. Đừng khử trùng điểm.

**Họ bài cùng pattern:** Kth Largest Element in an Array (215), Top K Frequent Elements (347), Find K Closest Elements (658), The K Weakest Rows in a Matrix (1337), Kth Smallest Element in a Sorted Matrix (378), Closest Binary Search Tree Value II (272, Premium).

## 15. Biến thể

1. **K điểm gần một điểm truy vấn `q` bất kỳ** (không chỉ gốc).
2. **K điểm xa nhất** (đối xứng: dùng min-heap hoặc sắp giảm).
3. **Khoảng cách của điểm gần thứ `k`** (chỉ cần một số): quickselect hoặc nhị phân trên khoảng cách.
4. **Dữ liệu dòng** (điểm tới lần lượt): duy trì heap kích thước `k`.

Cài đặt và kiểm chứng:

In [34]:
def k_closest_to(points, k, q):
    qx, qy = q
    return sorted(points, key=lambda p: (p[0] - qx) ** 2 + (p[1] - qy) ** 2)[:k]


def k_farthest(points, k):
    return sorted(points, key=d2, reverse=True)[:k]


def kth_smallest_distance2(points, k):
    dist = [d2(p) for p in points]
    lo, hi = 0, max(dist)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(d <= mid for d in dist) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


class KClosestStream:
    """Biến thể 4: nhận điểm từng cái một, luôn giữ k điểm gần gốc nhất."""

    def __init__(self, k):
        self.k, self.heap, self.n = k, [], 0

    def add(self, p):
        item = (-d2(p), self.n)
        self.n += 1
        entry = (item[0], item[1], tuple(p))
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, entry)
        elif entry > self.heap[0]:
            heapq.heapreplace(self.heap, entry)

    def current(self):
        return [list(e[2]) for e in self.heap]


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 30)
    pts = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(n)]
    k = rng.randint(1, n)
    q = [rng.randint(-5, 5), rng.randint(-5, 5)]
    shifted = [[p[0] - q[0], p[1] - q[1]] for p in pts]
    got = k_closest_to(pts, k, q)
    got_shifted = [[p[0] - q[0], p[1] - q[1]] for p in got]
    assert is_valid_k_closest(shifted, k, got_shifted)                              # (1) dịch gốc về q
    far = k_farthest(pts, k)
    assert sorted(d2(p) for p in far) == sorted(d2(p) for p in pts)[-k:]            # (2)
    assert kth_smallest_distance2(pts, k) == sorted(d2(p) for p in pts)[k - 1]      # (3)
    st = KClosestStream(k)
    for p in pts:
        st.add(p)
    assert is_valid_k_closest(pts, k, st.current())                                 # (4)
print("q bất kỳ, k xa nhất, khoảng cách thứ k, dữ liệu dòng: khớp oracle")

q bất kỳ, k xa nhất, khoảng cách thứ k, dữ liệu dòng: khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập `k` điểm | theo định nghĩa | `O(C(n,k)·n)` | `O(n)` | oracle `n ≲ 12` |
| 2. Sắp xếp | `sorted(key=d²)[:k]` | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. Max-heap `k` | giữ `k` tốt nhất | `O(n log k)` | `O(k)` | `k` nhỏ, dữ liệu dòng |
| 4. Quickselect | phân hoạch ba nhánh | `O(n)` tb | `O(n)` | nhanh nhất, không cần thứ tự |
| 5. Nhị phân khoảng cách | đếm `d² ≤ t` | `O(n log D)` | `O(n)` | đếm từng đợt |

Mọi cách cho kết quả hợp lệ trên toàn bộ lưới nhỏ (kể cả hòa) và trên `n` tới `10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [35]:
print(Solution().kClosest([[1, 3], [-2, 2]], 1), Solution().kClosest([[0, 0], [0, 0], [1, 1]], 2),
      sorted(Solution().kClosest([[1, 0], [0, 1], [-1, 0], [0, -1]], 4)), Solution().kClosest([[2, 2], [1, 1], [3, 3]], 1))

[[-2, 2]] [[0, 0], [0, 0]] [[-1, 0], [0, -1], [0, 1], [1, 0]] [[1, 1]]


### Bài 2 (tính tay)

`points = [[1,1],[2,0],[-1,-1],[0,3]]`, `k = 3`. (a) Tính `d²` từng điểm. (b) Tập 3 điểm gần nhất? (c) Có ai hòa ở biên không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện max-heap kích thước `k`:

In [36]:
def heap_todo(points, k):
    heap = []
    for i, p in enumerate(points):
        item = (-d2(p), i)
        pass            # TODO 1: nếu heap chưa đủ k thì heappush(item)
        pass            # TODO 2: ngược lại nếu item > heap[0] thì heapreplace(heap, item)
    return [points[i] for _, i in heap]


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(sorted(heap_todo(p, k)) == sorted(w) for (p, k), w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **sắp xếp theo `d²`**:

In [37]:
class MySolution:
    def kClosest(self, points, k):
        pass            # TODO


# Khi làm xong:
# assert all(sorted(MySolution().kClosest(p, k)) == sorted(w) for (p, k), w in examples)

### Bài 5 (mini challenge): K điểm gần điểm `q`

Viết `k_closest_to_ans(points, k, q)` trả về `k` điểm gần `q = (qx, qy)` nhất. Kiểm tra bằng cách **dịch gốc** về `q` và dùng `is_valid_k_closest`.

In [38]:
def k_closest_to_ans(points, k, q):
    pass            # TODO


# Khi làm xong:
# assert sorted(k_closest_to_ans([[0, 0], [5, 5], [4, 4], [10, 10]], 2, [5, 4])) == [[4, 4], [5, 5]]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: so `d²`, không cần `sqrt`.
- Mức 2: `[[0,0],[0,0],[1,1]]`, `k = 2` ⇒ hai điểm gốc (bản sao đều được giữ).
- Mức 3: `[[-2, 2]] [[0, 0], [0, 0]]`, bốn điểm (sau sắp xếp: `[[-1, 0], [0, -1], [0, 1], [1, 0]]`) và `[[1, 1]]`.

**Bài 2**

- Mức 1: `d² = 2, 4, 2, 9`.
- Mức 2: ba nhỏ nhất là `2, 2, 4`.
- Mức 3: `[[1,1],[-1,-1],[2,0]]`; không hòa ở biên vì `d²` thứ 3 (`4`) khác thứ 4 (`9`).

**Bài 3**

- Mức 1: `len(heap) < k`.
- Mức 2: `heapq.heappush(heap, item)`.
- Mức 3: `elif item > heap[0]: heapq.heapreplace(heap, item)`.

**Bài 4**

- Mức 1: khoá `lambda p: p[0]*p[0] + p[1]*p[1]`.
- Mức 2: `sorted(points, key=...)`.
- Mức 3: lấy `[:k]`.

**Bài 5**

- Mức 1: khoảng cách từ `p` tới `q` là `(px-qx)² + (py-qy)²`.
- Mức 2: dùng khoá đó trong `sorted`.
- Mức 3: `[:k]`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [39]:
vals = (Solution().kClosest([[1, 3], [-2, 2]], 1), Solution().kClosest([[0, 0], [0, 0], [1, 1]], 2),
        sorted(Solution().kClosest([[1, 0], [0, 1], [-1, 0], [0, -1]], 4)), Solution().kClosest([[2, 2], [1, 1], [3, 3]], 1))
print(vals)
assert vals == ([[-2, 2]], [[0, 0], [0, 0]], [[-1, 0], [0, -1], [0, 1], [1, 0]], [[1, 1]])
print("Bài 1 OK")

([[-2, 2]], [[0, 0], [0, 0]], [[-1, 0], [0, -1], [0, 1], [1, 0]], [[1, 1]])
Bài 1 OK


### Bài 2

In [40]:
pts = [[1, 1], [2, 0], [-1, -1], [0, 3]]
ds = [d2(p) for p in pts]
print(ds)
assert ds == [2, 4, 2, 9]
res = Solution().kClosest(pts, 3)
assert sorted(res) == [[-1, -1], [1, 1], [2, 0]] and is_valid_k_closest(pts, 3, res)
assert unique_answer(pts, 3) == sorted(res)                      # không hòa ở biên
print("Bài 2 OK:", sorted(res))

[2, 4, 2, 9]
Bài 2 OK: [[-1, -1], [1, 1], [2, 0]]


### Bài 3

In [41]:
def heap_answer(points, k):
    heap = []
    for i, p in enumerate(points):
        item = (-d2(p), i)
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
    return [points[i] for _, i in heap]


assert all(sorted(heap_answer(p, k)) == sorted(w) for (p, k), w in examples)
rng = random.Random(3)
for _ in range(100):
    pts = [[rng.randint(-4, 4), rng.randint(-4, 4)] for _ in range(rng.randint(1, 40))]
    k = rng.randint(1, len(pts))
    assert is_valid_k_closest(pts, k, heap_answer(pts, k))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [42]:
class MySolutionAnswer:
    def kClosest(self, points, k):
        return sorted(points, key=lambda p: p[0] * p[0] + p[1] * p[1])[:k]


assert all(sorted(MySolutionAnswer().kClosest(p, k)) == sorted(w) for (p, k), w in examples)
rng = random.Random(4)
for _ in range(100):
    pts = [[rng.randint(-4, 4), rng.randint(-4, 4)] for _ in range(rng.randint(1, 40))]
    k = rng.randint(1, len(pts))
    assert is_valid_k_closest(pts, k, MySolutionAnswer().kClosest(pts, k))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [43]:
def k_closest_to_ans(points, k, q):
    return sorted(points, key=lambda p: (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2)[:k]


assert sorted(k_closest_to_ans([[0, 0], [5, 5], [4, 4], [10, 10]], 2, [5, 4])) == [[4, 4], [5, 5]]
rng = random.Random(5)
for _ in range(200):
    pts = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(rng.randint(1, 30))]
    k = rng.randint(1, len(pts))
    q = [rng.randint(-6, 6), rng.randint(-6, 6)]
    shifted = [[p[0] - q[0], p[1] - q[1]] for p in pts]
    got = k_closest_to_ans(pts, k, q)
    assert is_valid_k_closest(shifted, k, [[p[0] - q[0], p[1] - q[1]] for p in got])
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chọn `k` nhỏ nhất theo một khoá):**

```python
class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        return sorted(points, key=lambda p: p[0] * p[0] + p[1] * p[1])[:k]
```

Heap kích thước `k` (khi `k` nhỏ): đẩy `(-d², i)`; nếu `len > k` hoặc phần tử mới gần hơn đỉnh thì `heapreplace`.

**Checklist trước khi submit:**

- [ ] So sánh `x*x + y*y` (không `sqrt`, không `x² − y²`, không Manhattan)?
- [ ] Trả **điểm** `[x, y]` (không phải khoảng cách), đúng `k` điểm?
- [ ] Không khử trùng điểm; bản sao toạ độ vẫn là các điểm riêng?
- [ ] Heap: đẩy `-d²` để giữ `k` điểm **nhỏ nhất**?
- [ ] Quickselect: phân hoạch **ba nhánh** cho trường hợp hòa?
- [ ] Biết đáp án có thể không duy nhất khi hòa ở biên (kiểm tra bằng hàm hợp lệ)?
- [ ] Đúng tên `class Solution`, `kClosest(self, points, k)`, không in debug.